In [45]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
import operator

In [46]:
load_dotenv()

True

In [47]:
model = ChatGroq(model='llama-3.3-70b-versatile')

In [48]:
class EvaluationSchema(BaseModel):
    feedback: str
    score: int = Field(description='Score out of 10', ge=0, le=10)

In [49]:
structured_model = model.with_structured_output(EvaluationSchema)

In [50]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]   #Reducer function is add 
    avg_score: float

In [51]:
def evaluate_language(state: UPSCState) -> UPSCState:

    prompt = f"Evaluate the language of the following essay:\n\n{state['essay']}\n\nProvide feedback and a score out of 10."

    output = structured_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [52]:
def evaluate_analysis(state: UPSCState) -> UPSCState:
    # Similar implementation to evaluate_language but focused on analysis
    prompt = f"Evaluate the analysis of the following essay:\n\n{state['essay']}\n\nProvide feedback and a score out of 10."
    
    output= structured_model.invoke(prompt)
    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [53]:
def evaluate_clarity(state: UPSCState) -> UPSCState:
    # Similar implementation to evaluate_language but focused on clarity
    prompt = f"Evaluate the clarity of the following essay:\n\n{state['essay']}\n\nProvide feedback and a score out of 10."
    
    output= structured_model.invoke(prompt)
    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [58]:
def final_evaluation(state: UPSCState) -> UPSCState:
    prompt = f"Based on the following feedback and scores, provide an overall evaluation of the essay:\n\nLanguage Feedback: {state['language_feedback']}\nAnalysis Feedback: {state['analysis_feedback']}\nClarity Feedback: {state['clarity_feedback']}\n\nProvide overall feedback and an average score out of 10."

    overall_feedback = model.invoke(prompt).content
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [59]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_clarity',evaluate_clarity)
graph.add_node('final_evaluation',final_evaluation)

In [60]:
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_clarity')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_clarity', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

initial_state = {'essay': "The capital of France is Paris. It is known for its rich history, culture, and landmarks such as the Eiffel Tower and the Louvre Museum."}

output = workflow.invoke(initial_state)
output

{'essay': 'The capital of France is Paris. It is known for its rich history, culture, and landmarks such as the Eiffel Tower and the Louvre Museum.',
 'language_feedback': 'The language used in the essay is simple and clear, but lacks complexity and variety. The sentences are short and basic, and there is no evidence of advanced vocabulary or grammatical structures. The essay could benefit from more descriptive language and nuanced expressions to convey a deeper understanding of the topic.',
 'analysis_feedback': "The essay provides a brief and accurate description of Paris, but it lacks depth and supporting details. The writing is simple and could be improved with more complex sentence structures and vocabulary. Additionally, the essay could benefit from more specific examples and analysis of the city's history, culture, and landmarks.",
 'clarity_feedback': "The essay is clear and concise, but lacks depth and supporting details. It effectively conveys basic information about Paris, b